In [1]:
pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Parquet File Reader

Quick notebook to read and explore parquet files in the data directory.

## Usage
1. Set the `file_path` variable to your parquet file
2. Optionally set `num_rows` and `columns_to_show`
3. Run all cells

In [2]:
import pandas as pd
from pathlib import Path

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

## Configuration

Update these variables with your file path and preferences:

**Note:** Since this notebook is in the `data/` directory, file paths should be relative to `data/`.
- ✅ Correct: `'options/etfs/SPY_20251011_205633.parquet'`
- ❌ Wrong: `'data/options/etfs/SPY_20251011_205633.parquet'`

In [3]:
# File to read (relative to data/ directory)
# Since this notebook is IN the data/ directory, just use the path from here
file_path = 'data/options/earnings/earnings_options_20251013.parquet'

# Number of rows to display
num_rows = 100

# Specific columns to show (leave as None to show all columns)
# Example: columns_to_show = ['symbol', 'strike', 'lastPrice', 'delta', 'gamma']
columns_to_show = None

## Load File

In [4]:
# Resolve file path
# Since this notebook is in the data/ directory, paths are relative to data/
file_path = Path(file_path)

# If the file doesn't exist as-is, it might already have 'data/' prefix - try without it
if not file_path.exists() and str(file_path).startswith('data'):
    # Remove the 'data/' prefix since we're already in the data directory
    parts = file_path.parts
    if parts[0] == 'data':
        file_path = Path(*parts[1:])

if not file_path.exists():
    # Try to find where we are and provide helpful error
    import os
    print(f"Current directory: {os.getcwd()}")
    print(f"Looking for: {file_path}")
    print(f"Absolute path: {file_path.absolute()}")
    raise FileNotFoundError(f"File not found: {file_path}\\n\\nMake sure the path is relative to the data/ directory.\\nExample: 'options/etfs/SPY_20251011_205633.parquet'")

print(f"Reading: {file_path}")

# Read parquet file
df = pd.read_parquet(file_path)

print(f"✓ Loaded {len(df):,} rows")

Reading: options\earnings\earnings_options_20251013.parquet
✓ Loaded 11,967 rows


## File Info

In [5]:
print(f"📊 File Info:")
print(f"  File: {file_path.name}")
print(f"  Total rows: {len(df):,}")
print(f"  Total columns: {len(df.columns)}")
print(f"  File size: {file_path.stat().st_size / 1024:.1f} KB")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")

📊 File Info:
  File: earnings_options_20251013.parquet
  Total rows: 11,967
  Total columns: 27
  File size: 740.2 KB
  Memory usage: 7.1 MB


## Column Information

In [6]:
print(f"📋 Columns ({len(df.columns)}):")
print()

col_info = []
for col in df.columns:
    dtype = df[col].dtype
    non_null = df[col].notna().sum()
    null_pct = (df[col].isna().sum() / len(df) * 100)
    col_info.append({
        'Column': col,
        'Type': str(dtype),
        'Non-Null': f"{non_null:,}",
        'Null %': f"{null_pct:.1f}%"
    })

col_df = pd.DataFrame(col_info)
display(col_df)

📋 Columns (27):



,Column,Type,Non-Null,Null %
0,symbol,object,"11,967",0.0%
1,expiration,datetime64[ns],"11,967",0.0%
2,optionType,object,"11,967",0.0%
3,contractSymbol,object,"11,967",0.0%
4,strike,float64,"11,967",0.0%
5,currency,object,"11,967",0.0%
6,lastPrice,float64,"11,967",0.0%
7,change,float64,"11,967",0.0%
8,percentChange,float64,"11,967",0.0%
9,volume,float64,"11,967",0.0%


## Data Preview

### Unique Values Summary

Shows the number of unique values in each column along with sample values.

In [35]:
# Get unique non-null snapshot_datetime values
unique_values = df['snapshot_datetime'].dropna().unique()
print(f"Unique snapshot_datetime values: {unique_values}")

unique_counts_df = df['snapshot_datetime'].value_counts(dropna=True).reset_index()
unique_counts_df.columns = ['snapshot_datetime', 'count']
display(unique_counts_df.head(10))

Unique snapshot_datetime values: <DatetimeArray>
['2025-10-13 17:15:54.787033', '2025-10-13 20:57:55.717720',
 '2025-10-13 21:40:18.259144']
Length: 3, dtype: datetime64[us]


,snapshot_datetime,count
0,2025-10-13 17:15:54.787033,3989
1,2025-10-13 20:57:55.717720,3989
2,2025-10-13 21:40:18.259144,3989


In [30]:
df['snapshot_datetime'].dropna().unique()

<DatetimeArray>
['2025-10-13 17:15:54.787033', '2025-10-13 20:57:55.717720',
 '2025-10-13 21:40:18.259144']
Length: 3, dtype: datetime64[us]

In [12]:
# Create a comprehensive unique values table
print("📊 Unique Values Summary")
print("=" * 80)

unique_data = []
for col in df.columns:
    unique_count = df[col].nunique()
    total_count = len(df[col])
    null_count = df[col].isna().sum()
    
    # Get sample unique values (up to 5)
    if unique_count <= 10:
        sample_values = df[col].dropna().unique()[:5]
        sample_str = ', '.join([str(v)[:30] for v in sample_values])
        if unique_count > 5:
            sample_str += f", ... (+{unique_count - 5} more)"
    else:
        sample_values = df[col].dropna().unique()[:3]
        sample_str = ', '.join([str(v)[:30] for v in sample_values])
        sample_str += f", ... (+{unique_count - 3} more)"
    
    unique_data.append({
        'Column': col,
        'Unique': unique_count,
        'Total': total_count,
        'Null': null_count,
        'Uniqueness': f"{(unique_count/total_count)*100:.1f}%",
        'Sample Values': sample_str if unique_count > 0 else 'N/A'
    })

unique_df = pd.DataFrame(unique_data)
display(unique_df)

print(f"\n✓ Showing unique value counts for {len(df.columns)} columns")

📊 Unique Values Summary


,Column,Unique,Total,Null,Uniqueness,Sample Values
0,symbol,413,11967,0,3.5%,"AA, AAL, AAP, ... (+410 more)"
1,expiration,8,11967,0,0.1%,"2025-10-24 00:00:00, 2025-10-31 00:00:00, 2025..."
2,optionType,2,11967,0,0.0%,"calls, puts"
3,contractSymbol,3989,11967,0,33.3%,"AA251024C00035500, AA251024P00035500, AAL25102..."
4,strike,539,11967,0,4.5%,"35.5, 12.0, 42.0, ... (+536 more)"
5,currency,1,11967,0,0.0%,USD
6,lastPrice,1810,11967,0,15.1%,"1.76, 1.2, 0.39, ... (+1807 more)"
7,change,1798,11967,0,15.0%,"0.0, 0.09000003, -0.15000004, ... (+1795 more)"
8,percentChange,3168,11967,0,26.5%,"0.0, 8.108111, -17.647062, ... (+3165 more)"
9,volume,620,11967,0,5.2%,"0.0, 2.0, 1322.0, ... (+617 more)"



✓ Showing unique value counts for 27 columns


### Sample Data

First N rows of the dataset (configurable via `num_rows`).

In [13]:
# Filter columns if specified
df_display = df.copy()

if columns_to_show:
    available_cols = [col for col in columns_to_show if col in df.columns]
    missing_cols = [col for col in columns_to_show if col not in df.columns]
    
    if missing_cols:
        print(f"⚠️  Missing columns: {', '.join(missing_cols)}")
    
    if available_cols:
        df_display = df_display[available_cols]
        print(f"✓ Showing only columns: {', '.join(available_cols)}")
    else:
        raise ValueError("None of the specified columns exist in the file")

print(f"\nFirst {min(num_rows, len(df_display))} rows:")
display(df_display.head(num_rows))


First 100 rows:


,symbol,expiration,optionType,contractSymbol,strike,currency,lastPrice,change,percentChange,volume,openInterest,bid,ask,contractSize,lastTradeDate,impliedVolatility,inTheMoney,snapshot_datetime,snapshot_date,snapshot_time,data_source,underlying_price,delta,gamma,theta,vega,rho
0,AA,2025-10-24,calls,AA251024C00035500,35.5,USD,1.76,0.00,0.000000,0.0,50.0,2.36,2.53,REGULAR,2025-10-10 19:58:19,0.702151,True,2025-10-13 17:15:54.787033,2025-10-13,17:15:54.787033,daily_eod,36.620,0.631559,0.088592,-0.000227,0.000229,5.701873e-05
1,AA,2025-10-24,puts,AA251024P00035500,35.5,USD,1.20,0.09,8.108111,2.0,10.0,1.08,1.18,REGULAR,2025-10-13 15:17:20,0.625004,False,2025-10-13 17:15:54.787033,2025-10-13,17:15:54.787033,daily_eod,36.620,-0.357967,0.098560,-0.000189,0.000226,-3.858995e-05
2,AAL,2025-10-24,calls,AAL251024C00012000,12.0,USD,0.39,0.00,0.000000,1322.0,3727.0,0.38,0.39,REGULAR,2025-10-13 16:40:36,0.609379,False,2025-10-13 17:15:54.787033,2025-10-13,17:15:54.787033,daily_eod,11.700,0.425301,0.332108,-0.000065,0.000076,1.268160e-05
3,AAL,2025-10-24,puts,AAL251024P00012000,12.0,USD,0.70,-0.15,-17.647062,850.0,1467.0,0.64,0.66,REGULAR,2025-10-13 15:54:12,0.566411,True,2025-10-13 17:15:54.787033,2025-10-13,17:15:54.787033,daily_eod,11.700,-0.583175,0.355763,-0.000056,0.000076,-2.033711e-05
4,AAP,2025-10-31,calls,AAP251031C00042000,42.0,USD,9.29,0.00,0.000000,0.0,3.0,9.45,11.90,REGULAR,2025-10-10 14:14:54,1.015630,True,2025-10-13 17:15:54.787033,2025-10-13,17:15:54.787033,daily_eod,53.005,0.881176,0.017099,-0.000198,0.000227,1.623925e-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,ABT,2025-10-17,calls,ABT251017C00150000,150.0,USD,0.01,-0.24,-96.000000,2.0,101.0,0.01,0.24,REGULAR,2025-10-13 15:30:43,0.588871,False,2025-10-13 17:15:54.787033,2025-10-13,17:15:54.787033,daily_eod,132.130,0.009580,0.003639,-0.000083,0.000031,1.021936e-06
96,ABT,2025-10-17,calls,ABT251017C00155000,155.0,USD,0.08,0.00,0.000000,3.0,3.0,0.00,0.20,REGULAR,2025-09-22 15:43:45,0.685550,False,2025-10-13 17:15:54.787033,2025-10-13,17:15:54.787033,daily_eod,132.130,0.005679,0.001972,-0.000061,0.000019,6.046639e-07
97,ABT,2025-10-17,puts,ABT251017P00080000,80.0,USD,0.06,0.00,0.000000,0.0,1.0,0.00,0.03,REGULAR,2025-10-06 15:15:03,1.656252,False,2025-10-13 17:15:54.787033,2025-10-13,17:15:54.787033,daily_eod,132.130,-0.000314,0.000058,-0.000010,0.000001,-3.547125e-08
98,ABT,2025-10-17,puts,ABT251017P00105000,105.0,USD,0.01,-0.15,-93.749990,1.0,18.0,0.01,0.05,REGULAR,2025-10-07 17:57:26,0.816408,False,2025-10-13 17:15:54.787033,2025-10-13,17:15:54.787033,daily_eod,132.130,-0.000824,0.000288,-0.000013,0.000003,-9.137878e-08


## Summary Statistics

In [8]:
# Show statistics for numeric columns
numeric_cols = df_display.select_dtypes(include=['number']).columns

if len(numeric_cols) > 0:
    print(f"Summary Statistics ({len(numeric_cols)} numeric columns):")
    display(df_display[numeric_cols].describe())
else:
    print("No numeric columns to display statistics for.")

Summary Statistics (15 numeric columns):


,strike,lastPrice,change,percentChange,volume,openInterest,bid,ask,impliedVolatility,underlying_price,delta,gamma,theta,vega,rho
count,11967.000000,11967.000000,11967.000000,11967.000000,11967.000000,11967.000000,11967.000000,11967.000000,11967.000000,11967.000000,11967.000000,11961.000000,11967.000000,11967.000000,11967.000000
mean,242.565244,20.446047,0.110807,-1.438947,86.834461,436.101780,19.030897,20.533004,0.790957,259.809894,0.086075,0.035715,-0.000877,0.000915,0.000029
std,332.944880,67.694833,6.537051,44.830626,381.275981,1532.591518,64.201689,65.654904,0.658469,350.858973,0.537683,0.064096,0.002468,0.001778,0.000732
min,0.500000,0.010000,-66.869995,-96.982765,0.000000,0.000000,0.000000,0.000000,0.000000,2.870000,-1.000000,0.000000,-0.087090,0.000000,-0.014345
25%,47.500000,0.700000,-0.100000,-9.243539,1.000000,10.000000,0.300000,0.960000,0.473638,50.720000,-0.317963,0.004219,-0.000589,0.000124,-0.000104
50%,115.000000,3.100000,0.000000,0.000000,5.000000,49.000000,2.570000,3.600000,0.638675,120.490000,0.015367,0.014322,-0.000225,0.000444,0.000003
75%,290.000000,11.400000,0.000000,0.000000,26.000000,250.000000,10.400000,11.900000,0.884034,340.640000,0.528623,0.037876,-0.000078,0.001069,0.000179
max,5190.000000,772.830000,302.800000,1959.999400,8118.000000,36142.000000,766.400000,774.000000,18.000004,5253.850000,1.000000,0.966467,0.000459,0.051717,0.019293


## Quick Analysis

Run custom analysis on the loaded dataframe:

In [9]:
# Example: Show unique values for categorical columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

if len(categorical_cols) > 0:
    print("Unique values in categorical columns:")
    for col in categorical_cols[:5]:  # Show first 5 categorical columns
        unique_count = df[col].nunique()
        print(f"\n{col}: {unique_count} unique values")
        if unique_count <= 10:
            print(f"  Values: {df[col].unique().tolist()}")

Unique values in categorical columns:

symbol: 413 unique values

optionType: 2 unique values
  Values: ['calls', 'puts']

contractSymbol: 3989 unique values

currency: 1 unique values
  Values: ['USD']

contractSize: 1 unique values
  Values: ['REGULAR']


## Custom Analysis

Add your own analysis here. The dataframe is available as `df`:

In [10]:
# Your custom analysis here
# Example for options data:

if 'optionType' in df.columns:
    print("Options Breakdown:")
    print(df['optionType'].value_counts())

if 'symbol' in df.columns:
    print("\nSymbols:")
    print(df['symbol'].value_counts())

if 'expiration' in df.columns:
    print("\nExpirations:")
    print(df['expiration'].value_counts().head(10))

Options Breakdown:
optionType
calls    6372
puts     5595
Name: count, dtype: int64

Symbols:
symbol
ASML    876
TSLA    792
BLK     528
INTC    387
RBLX    282
       ... 
CACI      3
GD        3
IDXX      3
SIMO      3
AMN       3
Name: count, Length: 413, dtype: int64

Expirations:
expiration
2025-11-21    4035
2025-10-24    3114
2025-10-17    2298
2025-10-31    1713
2025-11-07     762
2025-11-14      33
2026-03-20       6
2026-02-20       6
Name: count, dtype: int64


## Export Subset (Optional)

Uncomment to export filtered data to CSV:

In [11]:
# Uncomment to export
# output_file = 'exported_data.csv'
# df_display.head(num_rows).to_csv(output_file, index=False)
# print(f"✓ Exported to {output_file}")